In [ ]:
!pip install transformers==5.0.0 datasets accelerate -q


In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [ ]:
dataset = load_dataset("IbrahimAmin/egyptian-arabic-hate-speech")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/420k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/108k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6535 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1634 [00:00<?, ? examples/s]

In [ ]:
num_labels = 5
model_name = "UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

In [ ]:
unique_labels = sorted(dataset["train"].unique("label"))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print("Label mapping:", label2id)

def convert_labels(example):
    example['label'] = label2id[example['label']]
    return example

dataset = dataset.map(convert_labels)


Label mapping: {'Neutral': 0, 'Offensive': 1, 'Racism': 2, 'Religious Discrimination': 3, 'Sexism': 4}


Map:   0%|          | 0/6535 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Map:   0%|          | 0/1634 [00:00<?, ? examples/s]

In [ ]:
model.config.label2id = label2id
model.config.id2label = id2label
model.config.num_labels = num_labels

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/6535 [00:00<?, ? examples/s]

Map:   0%|          | 0/1634 [00:00<?, ? examples/s]

In [ ]:
import pandas as pd
pd.Series(dataset['train']['label']).value_counts()

,count
0,1708
1,1225
2,1201
4,1201
3,1200


In [ ]:
import pandas as pd
pd.Series(dataset['test']['label']).value_counts()

,count
0,427
1,306
4,301
2,300
3,300


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./filter_content_model",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Step,Training Loss
100,0.973500
200,0.326700
300,0.250100
400,0.222600
500,0.108400
600,0.110900
700,0.148000
800,0.095000
900,0.060600
1000,0.042100


TrainOutput(global_step=2045, training_loss=0.12544724700794826, metrics={'train_runtime': 412.1946, 'train_samples_per_second': 79.271, 'train_steps_per_second': 4.961, 'total_flos': 549866041390182.0, 'train_loss': 0.12544724700794826, 'epoch': 5.0})

In [ ]:
trainer.save_model("./filter_content_model")
tokenizer.save_pretrained("./filter_content_model")

metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.22769387066364288, 'eval_accuracy': 0.9614443084455324, 'eval_f1': 0.9613026886859912, 'eval_precision': 0.9614164374654226, 'eval_recall': 0.9614443084455324, 'eval_runtime': 3.6121, 'eval_samples_per_second': 452.37, 'eval_steps_per_second': 28.515, 'epoch': 5.0}


In [ ]:
sentiment_analyzer = pipeline("text-classification", model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment")


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0
